
# alpha len 与策略信号分析：从输入表到分析面板

这本 notebook 讲两个完整研究工作流：

1. 因子分析：把原始行情表变成 `date/code/factor/price`，再调用 `alphalen_analysis()`；
2. 策略分析：把原始行情表变成开平仓信号，再调用 `sig_stra_analysis()`。

这里会反复强调“输入列是什么、表达式做了什么、输出怎么看”。因为这两个分析算子本身很强，但如果输入列不对，面板就没有意义。


In [1]:
import sys
sys.path.insert(0, "/root/otters/otters-py/python")

import os
import importlib

import qust as qs
qs = importlib.reload(qs)

from qust import col, pms
from qust._polars import pl
import qust.datasource as qds

pl.Config.set_tbl_rows(16)
pl.Config.set_tbl_cols(14)

DATA_KLINE = "/root/qust-py/examples/data/data_kline2.parquet"
DATA_FUTURE = "/root/qust-py/examples/data/kline_data_all.parquet"
DATA_STOCK = "/root/qust-py/examples/data/stock_data_kline.parquet"

from qust import alpha101_assets, stra_assets



## 1. Alpha 输入规范：先把数据整理成标准四列

`alphalen_analysis` 不直接认识你的原始行情列，它只认标准分析输入：

| 列名 | 含义 | 例子 |
| --- | --- | --- |
| `date` | 横截面日期 | 2025-01-02 |
| `code` | 股票/资产代码 | SHSE.600000 |
| `factor` | 因子值 | alpha101 算出来的数值 |
| `price` | 用来算未来收益的价格 | close |

流程是：

```text
原始 K 线 -> alpha101 表达式生成 factor -> datetime 转 date -> close 复制为 price -> alphalen_analysis
```


In [2]:
stock_raw = pl.read_parquet(DATA_STOCK)
stock_dates = stock_raw.select(pl.col("datetime").dt.date().alias("date")).unique().sort("date").head(20)["date"].to_list()
stock_codes = stock_raw.select("code").unique().head(100)["code"].to_list()
stock_data = (
    stock_raw
    .with_columns(pl.col("datetime").dt.date().alias("_date"))
    .filter(pl.col("_date").is_in(stock_dates) & pl.col("code").is_in(stock_codes))
    .drop("_date")
)

alpha_exprs = alpha101_assets.get_all_alpha101_exprs()
alpha_factor_input = col(
    col.all,
    alpha_exprs[0].alias("factor"),
).with_cols(
    col("datetime").dt.date().alias("date"),
    col("close").alias("price"),
)

factor_data = alpha_factor_input.calc_data(stock_data)
print(stock_data.shape, factor_data.shape)
factor_data.select("date", "code", "factor", "price").head(12)


(1986, 7) (1986, 10)


date,code,factor,price
date,str,f64,f64
2025-01-02,"""SHSE.600023""",null,5.48
2025-01-02,"""SHSE.600155""",null,7.06
2025-01-02,"""SHSE.600169""",null,2.43
2025-01-02,"""SHSE.600190""",null,1.59
2025-01-02,"""SHSE.600333""",null,5.83
2025-01-02,"""SHSE.600408""",null,2.02
2025-01-02,"""SHSE.600489""",null,12.3
2025-01-02,"""SHSE.600529""",null,26.63
2025-01-02,"""SHSE.600540""",null,3.91



## 2. alphalen_analysis 参数：每个参数影响哪个面板

常用参数：

- `quantiles`：把因子分成几组，例如 5 组就是 Q1 到 Q5；
- `h1/h2/h3`：未来收益窗口，比如 1 日、5 日、10 日；
- `event_start/event_end`：事件分析窗口，观察事件前后收益形态。

这里用 `pms(...)` 而不是裸 int，是因为内部不仅要取当前值，还要生成参数表达式和列表表达式。对于分析面板这种组合算子，`Params` 能保留更多语义。


In [3]:
alphalen_expr = alpha_factor_input.alpha(
    quantiles=pms(2, 10).title("quantiles").value(5).step(1),
    h1=pms(1, 20).title("h1").value(1).step(1),
    h2=pms(1, 60).title("h2").value(5).step(1),
    h3=pms(1, 120).title("h3").value(10).step(1),
    event_start=pms(0, 20).title("event_start").value(5).step(1),
    event_end=pms(0, 30).title("event_end").value(10).step(1),
).alphalen_analysis()

print(alphalen_expr.display()[:600] + "...")


ExprAddGrid { inner: Expr { inner: ExprMakeMonitor { inner: Expr { inner: ExprPip { child: Expr { inner: ExprPip { child: Expr { inner: ExprPip { child: Expr { inner: ExprPip { child: Expr { inner: ExprPip { child: Expr { inner: ExprPip { child: Expr { inner: ExprPip { child: Expr { inner: ExprSelect { exprs: [Expr { inner: ExprPlExpr { inner: cs.all() }, metadata: RwLock { data: {}, poisoned: false, .. }, plugins: PluginRegistry { plugins: [] } }, Expr { inner: ExprPip { child: Expr { inner: ExprPip { child: Expr { inner: ExprPip { child: Expr { inner: ExprPip { child: Expr { inner: ExprPip {...


## 3. alphalen_analysis 出图

下面会打开 qust monitor。这个分析会包含 summary、quantile returns、IC、event study 等面板。


In [4]:
import contextlib
import io
import time

alphalen_runtime = alphalen_expr.runtime()
# 小样本下某些 monitor 子面板可能没有足够数据；plot 本身仍会返回 dashboard。
with contextlib.redirect_stderr(io.StringIO()):
    alphalen_runtime.plot(stock_data, open_in_jupyter=True, auto_open=False, height=900)
    time.sleep(0.5)


monitor session async start failed: Violin/Single needs at least 10, got 2 columns


## 4. returns_stats：收益统计表

`col("date", "pnl").bt.returns_stats()` 输入两列；如果有第三列 benchmark，会额外计算 Alpha/Beta/Benchmark Return。

这里先构造一个简单的日收益序列，展示输出格式。


In [5]:
stats_source = pl.DataFrame({
    "date": pl.date_range(pl.date(2025, 1, 1), pl.date(2025, 1, 30), interval="1d", eager=True),
    "pnl": [((i * 13) % 17 - 8) / 1000 for i in range(30)],
    "benchmark": [((i * 7) % 13 - 6) / 1200 for i in range(30)],
})

stats = col("date", "pnl", "benchmark").bt.returns_stats(periods_per_year=252).calc_data(stats_source)
stats


metric,value,value_float
str,str,f64
"""Start Index""","""2025-01-01""",null
"""End Index""","""2025-01-30""",null
"""Total Duration""","""29 days, 0:00:00""",null
"""Total Return [%]""","""-0.8325329185464581""",-0.832533
"""Benchmark Return [%]""","""-0.679286110002697""",-0.679286
"""Annualized Return [%]""","""-6.7816409019519135""",-6.781641
"""Annualized Volatility [%]""","""7.898109769497667""",7.89811
"""Max Drawdown [%]""","""1.5993007606286525""",1.599301
…,…,…



## 5. 策略信号输入规范：先生成开平仓信号

`sig_stra_analysis` 不是直接拿价格就能分析，它需要策略信号列：

| 列名 | 含义 |
| --- | --- |
| `open_long_sig` | 开多信号 |
| `exit_long_sig` | 平多信号 |
| `open_short_sig` | 开空信号 |
| `exit_short_sig` | 平空信号 |

再加上基础行情列：`datetime/ticker/close/open/high/low/volume`。

流程是：

```text
原始 K 线 -> stra_assets 内置策略生成信号 -> sig_stra_analysis 生成交易分析 dashboard
```


In [6]:
future_raw = pl.read_parquet(DATA_FUTURE)
future_data = future_raw.filter(pl.col("ticker") == "au").head(1_000)

stras = stra_assets.get_all_strategy_exprs()
stra = stras[0]

signals = col.with_cols(stra).calc_data(future_data)
signals.select(
    "datetime", "ticker", "close",
    "open_long_sig", "exit_long_sig", "open_short_sig", "exit_short_sig",
).head(12)


datetime,ticker,close,open_long_sig,exit_long_sig,open_short_sig,exit_short_sig
datetime[ms],str,f64,bool,bool,bool,bool
2022-07-02 00:01:00,"""au""",390.119995,false,false,false,false
2022-07-02 00:02:00.500,"""au""",390.140015,false,false,false,false
2022-07-02 00:03:00,"""au""",390.200012,false,false,false,false
2022-07-02 00:04:01,"""au""",390.160004,false,false,false,false
2022-07-02 00:05:00.500,"""au""",390.100006,false,false,false,false
2022-07-02 00:06:00.500,"""au""",390.140015,false,false,false,false
2022-07-02 00:07:00,"""au""",389.880005,false,false,false,false
2022-07-02 00:08:03,"""au""",389.880005,false,false,false,false
2022-07-02 00:09:00.500,"""au""",389.880005,false,false,false,false


## 6. sig_stra_analysis 出图

`bt.sig_stra_analysis()` 会根据开平仓信号生成交易散点、持仓收益、分组 pnl 和总 pnl monitor。


In [7]:
sig_analysis = col.with_cols(stra).select(
    col.all.bt.sig_stra_analysis()
).runtime()

sig_analysis.plot(future_data, open_in_jupyter=True, auto_open=False, height=900)



## 7. trade_row / bin_trade / violin profile：拆开 dashboard 自己做分析

完整 dashboard 很方便，但研究时经常要拆开某个问题。

下面这段表达式做的是：

1. 用策略生成信号；
2. `bin_trade()` 给每一笔交易编号；
3. 在每笔交易内部计算已经持有多少 bar：`open_elapsed`；
4. 在每笔交易内部累计收益：`ret_cum`；
5. 对每个 `open_elapsed` 的 `ret_cum` 分布做 violin profile。

这就是“完整分析面板 -> 拆成可复用小分析”的思路。


In [8]:
trade_profile = (
    col
    .with_cols(
        stra,
        (col("close").pct().expanding()).alias("ret"),
    )
    .with_cols(
        col("open_long_sig", "exit_long_sig").stra.bin_trade().alias("bin_trade")
    )
    .over("ticker")
    .filter(col("bin_trade").is_not_null())
    .with_cols(
        col(
            col("ret").count().expanding().alias("open_elapsed"),
            col("ret").sum().expanding().alias("ret_cum"),
        ).over("bin_trade")
    )
    .filter(col("ret_cum").is_not_null())
    .select(
        col("ret_cum")
        .batch.violin_profile(lower_bound=0.05, up_bound=0.95)
        .group_by("open_elapsed")
        .batch.sort("open_elapsed")
    )
    .calc_data(future_data)
)

trade_profile.head(12)


open_elapsed,count,min,q1,median,q3,max,y,half_width,is_point
u32,u64,f64,f64,f64,f64,f64,list[f64],list[f64],bool
1,1,0.000307,0.000307,0.000307,0.000307,0.000307,[],[],true
2,1,0.000563,0.000563,0.000563,0.000563,0.000563,[],[],true
3,1,0.000563,0.000563,0.000563,0.000563,0.000563,[],[],true
4,1,0.000563,0.000563,0.000563,0.000563,0.000563,[],[],true
5,1,0.000307,0.000307,0.000307,0.000307,0.000307,[],[],true
6,1,0.00046,0.00046,0.00046,0.00046,0.00046,[],[],true
7,1,0.00046,0.00046,0.00046,0.00046,0.00046,[],[],true
8,1,0.00046,0.00046,0.00046,0.00046,0.00046,[],[],true
9,1,0.000409,0.000409,0.000409,0.000409,0.000409,[],[],true


## 8. 组合建议

常见研究流程：

1. 用 qust 表达式生成 factor 或 strategy signals；
2. 用 `alphalen_analysis` 或 `sig_stra_analysis` 快速看整体面板；
3. 对发现的问题拆成更小的表达式，例如 returns_stats、trade_row、violin_profile；
4. 用 `pms/opt_params/optuna_params` 做参数扫描；
5. 最后把稳定表达式保存并用于更大样本或流式数据源。
